#  CASE 02 — Square 6×6 + 3×3 — Correlation Center

In [1]:
%%writefile case02_sq_3x3_corrCenter.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define WIDTH       6
#define HEIGHT      6
#define MASK_WIDTH  3
#define MASK_HEIGHT 3
#define BLOCK_SIZE  8

__global__ void correlationCenter(int *dA, int *dMask, int *dC,
                                  int width, int height,
                                  int mWidth, int mHeight)
{
    int col = threadIdx.x + blockIdx.x * blockDim.x;
    int row = threadIdx.y + blockIdx.y * blockDim.y;
    if (row < height && col < width)
    {
        int sum = 0;
        for (int i = 0; i < mHeight; i++)
            for (int j = 0; j < mWidth; j++)
            {
                int r = row + i - mHeight/2;  /* center: subtract half */
                int c = col + j - mWidth/2;
                if (r>=0 && r<height && c>=0 && c<width)
                    sum += dA[r*width+c] * dMask[i*mWidth+j];
            }
        dC[row*width+col] = sum;
    }
}

void printMatrix(const char *label, int *M, int w, int h)
{
    printf("\n%s:\n", label);
    for (int r = 0; r < h; r++)
    {
        for (int c = 0; c < w; c++)
            printf("%6d", M[r*w+c]);
        printf("\n");
    }
}

int main()
{
    int size     = WIDTH * HEIGHT * sizeof(int);
    int maskSize = MASK_WIDTH * MASK_HEIGHT * sizeof(int);

    int *hA    = (int*) malloc(size);
    int *hMask = (int*) malloc(maskSize);
    int *hC    = (int*) malloc(size);

    srand(time(NULL));
    for (int i = 0; i < WIDTH*HEIGHT; i++)
        hA[i] = rand()%9+1;

    int tempMask[3][3] = {{1,0,-1},{2,0,-2},{1,0,-1}};
    for (int i = 0; i < MASK_HEIGHT; i++)
        for (int j = 0; j < MASK_WIDTH; j++)
            hMask[i*MASK_WIDTH+j] = tempMask[i][j];

    printMatrix("Input Matrix (6x6)", hA,    WIDTH,      HEIGHT);
    printMatrix("Mask (3x3)",         hMask, MASK_WIDTH, MASK_HEIGHT);

    int *dA, *dMask, *dC;
    cudaMalloc((void**)&dA,    size);
    cudaMalloc((void**)&dMask, maskSize);
    cudaMalloc((void**)&dC,    size);
    cudaMemcpy(dA,    hA,    size,     cudaMemcpyHostToDevice);
    cudaMemcpy(dMask, hMask, maskSize, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 DimGrid((int)ceil((float)WIDTH/BLOCK_SIZE),
                 (int)ceil((float)HEIGHT/BLOCK_SIZE), 1);

    cudaEvent_t start, stop; float gpuTime;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    correlationCenter<<<DimGrid,DimBlock>>>(dA,dMask,dC,
                       WIDTH,HEIGHT,MASK_WIDTH,MASK_HEIGHT);

    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    printMatrix("OUTPUT: Correlation Center (6x6, 3x3)", hC, WIDTH, HEIGHT);
    printf("\nGPU Time: %.4f ms\n", gpuTime);
    printf("Grid: %dx%d  Block: %dx%d\n",
            DimGrid.x,DimGrid.y,DimBlock.x,DimBlock.y);

    cudaFree(dA); cudaFree(dMask); cudaFree(dC);
    free(hA); free(hMask); free(hC);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

Writing case02_sq_3x3_corrCenter.cu


In [2]:
!nvcc -arch=sm_75 case02_sq_3x3_corrCenter.cu -o case02_sq_3x3_corrCenter

!./case02_sq_3x3_corrCenter


Input Matrix (6x6):
     6     2     1     8     5     9
     2     3     9     6     7     6
     5     4     2     2     2     3
     4     8     6     4     3     6
     6     4     5     7     2     3
     8     8     4     6     4     6

Mask (3x3):
     1     0    -1
     2     0    -2
     1     0    -1

OUTPUT: Correlation Center (6x6, 3x3):
    -7     3   -15    -6    -2    17
   -12    -6   -10     0    -2    21
   -19    -3     5     5    -4    14
   -24     0     7     9    -1    10
   -24     4     0     9     6    11
   -20     9     1     3     4    10

GPU Time: 0.1044 ms
Grid: 1x1  Block: 8x8
